In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [14]:
power = pd.read_csv("data/power.csv")
power["t"] = pd.to_datetime(power["t"], unit="s")
power = power.set_index("t")
power.head()

,value
t,
2026-10-03 17:54:19.166879892,251.476974
2026-10-03 17:54:19.290238857,243.439469
2026-10-03 17:54:19.395206928,252.387084
2026-10-03 17:54:19.500396013,255.623387
2026-10-03 17:54:19.603703976,250.526138


In [15]:
power.describe()

,value
count,5622.000000
mean,273.492831
std,67.600152
min,-50.000000
25%,225.326670
50%,290.913359
75%,329.020249
max,377.724566


In [16]:
def load(name):
    df = pd.read_csv(f"data/{name}.csv")
    df["t"] = pd.to_datetime(df["t"], unit="s")
    return df.set_index("t")

power = load("power")
hr = load("hr")
cadence = load("cadence")
speed = load("speed")
gps = load("gps")

for name, df in [("power", power), ("hr", hr), ("cadence", cadence), ("speed", speed), ("gps", gps)]:
    print(name, len(df))

power 5622
hr 585
cadence 585
speed 585
gps 585


In [17]:
power_raw = power.copy()
hr_raw = hr.copy()

In [18]:
n = len(power)
power = power[power["value"] >= 0]
print("power removed:", n - len(power))

n = len(hr)
hr = hr[hr["value"] <= 220]
print("hr removed:", n - len(hr))

power removed: 49
hr removed: 3


In [19]:
print("cadence missing:", cadence["value"].isna().sum())

n = len(speed)
speed = speed[speed["value"] > 0]
print("speed removed:", n - len(speed))

cadence missing: 5
speed removed: 6


In [20]:
median_lat = gps["lat"].rolling(5, center=True, min_periods=1).median()
jump_m = (gps["lat"] - median_lat).abs() * 111000
n = len(gps)
gps = gps[jump_m < 50]
print("gps removed:", n - len(gps))

gps removed: 6


In [22]:
df = pd.concat({
    "power": power["value"].resample("1s").mean(),
    "hr": hr["value"].resample("1s").mean(),
    "cadence": cadence["value"].resample("1s").mean(),
    "speed": speed["value"].resample("1s").mean(),
    "lat": gps["lat"].resample("1s").mean(),
    "lon": gps["lon"].resample("1s").mean(),
}, axis=1)

print("missing before:\n", df.isna().sum())
df = df.interpolate()
print("missing after:\n", df.isna().sum())
df.head()

missing before:
 power      109
hr         116
cadence    118
speed      119
lat        119
lon        119
dtype: int64
missing after:
 power      0
hr         0
cadence    0
speed      0
lat        0
lon        0
dtype: int64


,power,hr,cadence,speed,lat,lon
t,,,,,,
2026-10-03 17:54:19,251.260549,120.536125,92.580372,39.973473,45.040100,7.4
2026-10-03 17:54:20,250.546249,124.869518,91.941471,40.998387,45.040231,7.4
2026-10-03 17:54:21,261.356051,123.009780,89.693934,39.766842,45.040301,7.4
2026-10-03 17:54:22,256.887649,122.763559,87.978258,40.461690,45.040441,7.4
2026-10-03 17:54:23,271.053238,125.246215,85.799445,41.156539,45.040558,7.4
